# Entry timing: BTC/USDT continuation setup on 15-minute data

**Research question.** After a daily continuation signal, does waiting before buying give a better or worse entry price than buying immediately at the next day's open?

## Pre-registered design

The design below was decided before any results were computed and is followed exactly. No delay, holding period, cost, rule or threshold is changed in response to the results.

**Data**
1. 15-minute BTC/USDT history is downloaded once with `refresh_symbol(exchange, "BTC/USDT", DATA_DIR, exchange.milliseconds(), timeframe="15m")`, where `exchange = ccxt.binance()`.
2. It is loaded with `load_processed(..., timeframe="15m")`; the number of candles, the date range and the 10 largest gaps are reported.
3. Daily BTC/USDT is loaded with `load_processed` (default `"1d"`), then passed through `add_features` and `add_regimes`.
4. Sanity check: each day's daily open is compared with the open of that day's 00:00 UTC 15-minute candle (match = within 0.01%).

**Signals and trades**

5. Signals come from `setup_flag(daily, "continuation")`, with `SETUPS` unchanged.
6. Trades are the Phase 6 trades: `find_trades(signal, hold_days)` on the daily data, for `hold_days` = 5 and 10. A signal at the close of day t enters on day t+1.

**Entry delays**

7. Delays of 0, 1, 4 and 12 hours after 00:00 UTC on the entry day. The entry price is the open of the 15-minute candle at that timestamp. Delay 0 is the baseline.
8. If the 15-minute candle at a delay's timestamp is missing, that trade is skipped for that delay only; skipped trades are counted per delay.
9. The exit is fixed for all delays: the daily open on the exit day from `find_trades`. Only the entry changes.

**Measurements**

10. For each trade and delay: entry improvement = log(baseline entry price / delayed entry price), so a positive value means the delayed entry was cheaper. Net return per trade with `cost_per_side=0.0015`, using that delay's entry and the fixed exit.
11. For each holding period and delay: number of trades, mean and median entry improvement, share of trades with positive improvement, standard error of the mean, and mean net return.
12. Base rates: the same entry improvement for each delay on (a) every day and (b) every day whose trend regime was "up" on the signal day, treating each day as a hypothetical signal. Each base rate's mean is reported, together with the difference between the signal trades' mean and the base rate in standard errors (using the signal trades' standard error).

**Periods**

13. Development: trades whose exit date is on or before 2023-12-31. Held-out test: trades whose entry date is on or after 2024-01-01. Base rates use the same periods. Development and test results are reported in separate tables.

No predictions were recorded before these results were computed.

## Step 0: Imports and settings

`HOLDS` and `COST` are the values used in notebooks 06–08. `DELAYS_H` are the pre-registered entry delays in hours.

In [1]:
%load_ext autoreload
%autoreload 2

import time
from pathlib import Path

import ccxt
import numpy as np
import pandas as pd

from cryptosense.data import refresh_symbol, load_processed, timeframe_to_ms
from cryptosense.features import add_features
from cryptosense.regimes import add_regimes
from cryptosense.setups import SETUPS, setup_flag
from cryptosense.backtest import find_trades

DATA_DIR = Path.cwd().parent / "data"
SYMBOL = "BTC/USDT"
HOLDS = [5, 10]
DELAYS_H = [0, 1, 4, 12]
COST = 0.0015
MATCH_TOL = 0.0001  # 0.01%
DEV_END = pd.Timestamp("2023-12-31", tz="UTC")
TEST_START = pd.Timestamp("2024-01-01", tz="UTC")
HOUR_MS = 3_600_000
CANDLE_15M_MS = timeframe_to_ms("15m")

## Step 1: Download 15-minute BTC/USDT history

Run once. `refresh_symbol` fetches the full history, validates it, drops the incomplete candle and saves raw and processed files. If the download fails, the notebook stops here.

In [2]:
exchange = ccxt.binance()
start = time.perf_counter()
result = refresh_symbol(exchange, SYMBOL, DATA_DIR, exchange.milliseconds(), timeframe="15m")
elapsed = time.perf_counter() - start

print(result)
print(f"Download took {elapsed:.1f} s ({elapsed / 60:.1f} min)")

{'symbol': 'BTC/USDT', 'rows': 321296, 'last_date': datetime.date(2026, 10, 21), 'problems': {'gaps': 33, 'missing_days': 565, 'zero_volume_days': 56}}
Download took 175.3 s (2.9 min)


## Step 2: Load the 15-minute data and inspect gaps

A gap is a step between consecutive candles longer than 15 minutes. "Gap ended" is the timestamp of the first candle after the gap.

In [3]:
m15 = load_processed(SYMBOL, DATA_DIR, timeframe="15m")
print(f"candles: {len(m15):,}")
print(f"range:   {m15.index[0]} -> {m15.index[-1]}")

steps = m15["timestamp"].diff()
gap_steps = steps[steps > CANDLE_15M_MS]
missing = (gap_steps // CANDLE_15M_MS - 1).astype(int)
print(f"gaps: {len(gap_steps)} | missing candles in total: {missing.sum():,}")

gaps = pd.DataFrame({
    "gap_ended": gap_steps.index,
    "missing_candles": missing.to_numpy(),
})
gaps.nlargest(10, "missing_candles").reset_index(drop=True)

candles: 321,296
range:   2017-08-17 04:00:00+00:00 -> 2026-10-21 21:00:00+00:00
gaps: 33 | missing candles in total: 565


,gap_ended,missing_candles
0,2018-02-09 10:00:00+00:00,134
1,2018-06-26 12:00:00+00:00,40
2,2019-05-15 13:00:00+00:00,40
3,2019-08-15 10:00:00+00:00,32
4,2018-07-04 08:00:00+00:00,30
5,2018-11-14 09:00:00+00:00,28
6,2017-09-06 23:00:00+00:00,27
7,2019-03-12 08:00:00+00:00,24
8,2020-02-19 17:30:00+00:00,23
9,2021-04-25 08:45:00+00:00,18


## Step 3: Load daily data, add features and regimes

In [4]:
daily = add_regimes(add_features(load_processed(SYMBOL, DATA_DIR)))
print(f"daily: {daily.index[0].date()} -> {daily.index[-1].date()} | {len(daily):,} days")

# 15-minute opens looked up by integer candle timestamp (ms, UTC)
open_15m = m15.set_index("timestamp")["open"]

daily: 2017-08-17 -> 2026-10-20 | 3,352 days


## Step 4: Sanity check — daily open vs the 00:00 UTC 15-minute open

For each day, the daily open is compared with the open of the 15-minute candle starting at 00:00 UTC that day. A match means a relative difference of at most 0.01%.

In [5]:
midnight_open = open_15m.reindex(daily["timestamp"]).to_numpy()
rel_diff = np.abs(midnight_open / daily["open"].to_numpy() - 1)
has_candle = ~np.isnan(midnight_open)

check = pd.Series({
    "days": len(daily),
    "match (<= 0.01%)": int((has_candle & (rel_diff <= MATCH_TOL)).sum()),
    "differ (> 0.01%)": int((has_candle & (rel_diff > MATCH_TOL)).sum()),
    "no 00:00 candle": int((~has_candle).sum()),
})
print(check.to_string())

differing = pd.DataFrame({
    "daily_open": daily["open"],
    "open_15m_0000": midnight_open,
    "rel_diff": rel_diff,
})[has_candle & (rel_diff > MATCH_TOL)]
no_candle_days = daily.index[~has_candle]
print("\nDays without a 00:00 candle:", [d.date() for d in no_candle_days])
differing.head(20)

days                3352
match (<= 0.01%)    3350
differ (> 0.01%)       0
no 00:00 candle        2

Days without a 00:00 candle: [datetime.date(2017, 8, 17), datetime.date(2018, 2, 9)]


,daily_open,open_15m_0000,rel_diff
date,,,


## Step 5: Continuation signal

`setup_flag` is used exactly as defined; the rule is printed from `SETUPS` unchanged.

In [6]:
print("continuation rule:", SETUPS["continuation"])
signal = setup_flag(daily, "continuation")
print(f"signal days: {int(signal.sum())}")

continuation rule: [('mom_20', '>', 0.0), ('er_20', '>', 0.4), ('rel_volume_20', '>', 1.5)]
signal days: 102


## Step 6: Trades (Phase 6 rules)

`find_trades(signal, hold_days)` on the daily data, for 5- and 10-day holds. Each trade records its signal day (t), entry day (t+1), exit day and the fixed exit price (the daily open on the exit day). Periods: development = exit on or before 2023-12-31; test = entry on or after 2024-01-01.

In [7]:
def assign_period(frame):
    """Development if the exit is on or before DEV_END; test if the entry is on or after TEST_START."""
    period = pd.Series(pd.NA, index=frame.index, dtype="object")
    period[frame["exit_date"] <= DEV_END] = "development"
    period[frame["entry_date"] >= TEST_START] = "test"
    return period


def trade_frame(entry_pos, exit_pos):
    """Signal, entry and exit dates plus the fixed exit price for the given row positions."""
    entry_pos, exit_pos = np.asarray(entry_pos), np.asarray(exit_pos)
    frame = pd.DataFrame({
        "signal_date": daily.index[entry_pos - 1],
        "entry_date": daily.index[entry_pos],
        "exit_date": daily.index[exit_pos],
        "entry_ts": daily["timestamp"].to_numpy()[entry_pos],
        "exit_price": daily["open"].to_numpy()[exit_pos],
        "signal_trend": daily["trend"].to_numpy()[entry_pos - 1],
    })
    frame["period"] = assign_period(frame)
    return frame


trades = {}
for hold in HOLDS:
    pairs = find_trades(signal, hold)
    trades[hold] = trade_frame([e for e, _ in pairs], [x for _, x in pairs])

pd.DataFrame({
    f"{hold}d": trades[hold]["period"].fillna("spans boundary").value_counts()
    for hold in HOLDS
}).fillna(0).astype(int)

,5d,10d
period,,
development,36,27
test,12,11


## Step 7: Entry prices at each delay

The entry price for a delay of h hours is the open of the 15-minute candle at 00:00 UTC + h on the entry day. A missing candle gives no price (NaN).

In [8]:
def add_entries(frame):
    """Add the entry price for each delay, entry improvement vs delay 0, and net return."""
    frame = frame.copy()
    for d in DELAYS_H:
        ts = frame["entry_ts"] + d * HOUR_MS
        frame[f"entry_{d}h"] = open_15m.reindex(ts).to_numpy()
    for d in DELAYS_H:
        entry = frame[f"entry_{d}h"]
        frame[f"improvement_{d}h"] = np.log(frame["entry_0h"] / entry)
        frame[f"net_{d}h"] = (frame["exit_price"] * (1 - COST)) / (entry * (1 + COST)) - 1
    return frame


for hold in HOLDS:
    trades[hold] = add_entries(trades[hold])

trades[5].head()

,signal_date,entry_date,exit_date,entry_ts,exit_price,signal_trend,period,entry_0h,entry_1h,entry_4h,entry_12h,improvement_0h,net_0h,improvement_1h,net_1h,improvement_4h,net_4h,improvement_12h,net_12h
0,2017-10-12 00:00:00+00:00,2017-10-13 00:00:00+00:00,2017-10-18 00:00:00+00:00,1507852800000,5595.00,NaN,development,5439.99,5407.74,5740.02,5658.97,0.0,0.025414,0.005946,0.031529,-0.053685,-0.028185,-0.039465,-0.014266
1,2017-10-18 00:00:00+00:00,2017-10-19 00:00:00+00:00,2017-10-24 00:00:00+00:00,1508371200000,5909.47,NaN,development,5513.00,5498.98,5570.24,5710.00,0.0,0.068705,0.002546,0.071429,-0.010329,0.057722,-0.035110,0.031833
2,2017-10-24 00:00:00+00:00,2017-10-25 00:00:00+00:00,2017-10-30 00:00:00+00:00,1508889600000,6133.01,NaN,development,5506.92,5476.16,5515.93,5469.99,0.0,0.110355,0.005601,0.116592,-0.001635,0.108542,0.006729,0.117852
3,2017-11-08 00:00:00+00:00,2017-11-09 00:00:00+00:00,2017-11-14 00:00:00+00:00,1510185600000,6465.99,NaN,development,7303.01,7241.95,7277.94,7076.00,0.0,-0.117265,0.008396,-0.109822,0.003439,-0.114224,0.031578,-0.088946
4,2017-11-30 00:00:00+00:00,2017-12-01 00:00:00+00:00,2017-12-06 00:00:00+00:00,1512086400000,11699.99,up,development,9837.00,9685.02,9666.28,9960.01,0.0,0.185823,0.015570,0.204431,0.017507,0.206766,-0.012427,0.171178


## Step 8: Skipped trades per delay

A trade is skipped for a delay when the 15-minute candle at that delay's timestamp is missing. "No baseline" counts trades whose delay-0 candle is missing, for which no entry improvement can be computed at any delay.

In [9]:
rows = []
for hold in HOLDS:
    for period in ["development", "test"]:
        t = trades[hold][trades[hold]["period"] == period]
        for d in DELAYS_H:
            rows.append({
                "hold": hold,
                "period": period,
                "delay_h": d,
                "trades": len(t),
                "skipped (candle missing)": int(t[f"entry_{d}h"].isna().sum()),
                "no baseline": int(t["entry_0h"].isna().sum()),
            })
skipped = pd.DataFrame(rows).set_index(["hold", "period", "delay_h"])
skipped

trades  skipped (candle missing)  no baseline
hold period      delay_h                                               
5    development 0            36                         0            0
                 1            36                         0            0
                 4            36                         0            0
                 12           36                         0            0
     test        0            12                         0            0
                 1            12                         0            0
                 4            12                         0            0
                 12           12                         0            0
10   development 0            27                         0            0
                 1            27                         0            0
                 4            27                         1            0
                 12           27                         1            0
     test        0            11                         0            0
                 1            11                         0            0
                 4            11                         0            0
                 12           11                         0            0

## Step 9: Base rates — every day as a hypothetical signal

Every daily row t is treated as a signal (entry on day t+1, exit on day t+1+hold), with the same delays, the same entry-improvement measure and the same periods. Pool (a) uses every day; pool (b) uses days whose trend regime was "up" on the signal day.

In [10]:
base = {}
for hold in HOLDS:
    entry_pos = np.arange(1, len(daily) - hold)
    base[hold] = add_entries(trade_frame(entry_pos, entry_pos + hold))

print({hold: len(frame) for hold, frame in base.items()})

{5: 3346, 10: 3341}


## Step 10: Measurements by period

For each holding period and delay: number of trades used, mean and median entry improvement, share of trades with a positive improvement, standard error of the mean, and mean net return (cost 0.15% per side). Entry improvement and returns are in percent (entry improvement is a log difference × 100).

Base rates: mean entry improvement over every day and over up-trend days, and the difference between the signal trades' mean and each base rate, divided by the signal trades' standard error. For delay 0 the improvement is 0 by definition, so its standard error is 0 and the difference in standard errors is undefined (NaN).

In [11]:
def summarize(t, base_frame, d):
    """Entry-improvement statistics for one delay, with base rates from the same period."""
    impr = t[f"improvement_{d}h"].dropna()
    net = t[f"net_{d}h"].dropna()
    se = impr.std() / np.sqrt(len(impr)) if len(impr) > 1 else np.nan
    base_all = base_frame[f"improvement_{d}h"].dropna()
    base_up = base_frame.loc[base_frame["signal_trend"] == "up", f"improvement_{d}h"].dropna()
    in_se = lambda b: (impr.mean() - b.mean()) / se if se > 0 else np.nan
    return {
        "trades": len(impr),
        "mean_impr_%": 100 * impr.mean(),
        "median_impr_%": 100 * impr.median(),
        "share_positive": (impr > 0).mean(),
        "se_%": 100 * se,
        "mean_net_return_%": 100 * net.mean(),
        "net_trades": len(net),
        "base_all_mean_%": 100 * base_all.mean(),
        "base_all_days": len(base_all),
        "diff_vs_all_in_se": in_se(base_all),
        "base_up_mean_%": 100 * base_up.mean(),
        "base_up_days": len(base_up),
        "diff_vs_up_in_se": in_se(base_up),
    }


def period_table(period):
    rows = []
    for hold in HOLDS:
        t = trades[hold][trades[hold]["period"] == period]
        b = base[hold][base[hold]["period"] == period]
        for d in DELAYS_H:
            rows.append({"hold": hold, "delay_h": d, **summarize(t, b, d)})
    return pd.DataFrame(rows).set_index(["hold", "delay_h"])


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
SIGNAL_COLS = ["trades", "mean_impr_%", "median_impr_%", "share_positive", "se_%", "mean_net_return_%", "net_trades"]
BASE_COLS = ["mean_impr_%", "se_%", "base_all_mean_%", "base_all_days", "diff_vs_all_in_se",
             "base_up_mean_%", "base_up_days", "diff_vs_up_in_se"]
dev_table = period_table("development")
test_table = period_table("test")

### Development period (exit on or before 2023-12-31)

In [12]:
print("Signal trades:")
print(dev_table[SIGNAL_COLS].round(4).to_string())
print("\nBase rates:")
print(dev_table[BASE_COLS].round(4).to_string())

Signal trades:
              trades  mean_impr_%  median_impr_%  share_positive    se_%  mean_net_return_%  net_trades
hold delay_h                                                                                           
5    0            36       0.0000         0.0000          0.0000  0.0000             2.8643          36
     1            36      -0.0961         0.0521          0.5556  0.1529             2.7566          36
     4            36      -0.3704        -0.1255          0.4167  0.2844             2.4472          36
     12           36       0.1683        -0.3248          0.3889  0.6139             2.9209          36
10   0            27       0.0000         0.0000          0.0000  0.0000             6.1020          27
     1            27      -0.0719         0.0828          0.5926  0.1809             6.0324          27
     4            26      -0.0385         0.1348          0.5385  0.4095             6.2838          26
     12           26       0.1718        -0.2474 

### Held-out test period (entry on or after 2024-01-01)

In [13]:
print("Signal trades:")
print(test_table[SIGNAL_COLS].round(4).to_string())
print("\nBase rates:")
print(test_table[BASE_COLS].round(4).to_string())

Signal trades:
              trades  mean_impr_%  median_impr_%  share_positive    se_%  mean_net_return_%  net_trades
hold delay_h                                                                                           
5    0            12       0.0000         0.0000          0.0000  0.0000             2.8271          12
     1            12       0.0313         0.0668          0.5833  0.1177             2.8493          12
     4            12      -0.2420        -0.0459          0.5000  0.2467             2.5537          12
     12           12      -0.3496         0.3226          0.5833  0.6346             2.4091          12
10   0            11       0.0000         0.0000          0.0000  0.0000             4.8844          11
     1            11       0.0059         0.0386          0.5455  0.1259             4.8866          11
     4            11      -0.2815        -0.1781          0.4545  0.2668             4.5782          11
     12           11      -0.5956         0.2942 

## Results

_To be filled in with observations only, after the notebook has run._

## Interpretation

**The baseline is the Phase 6 entry.** On 3,350 of 3,352 days the 00:00 UTC 15-minute open matched the daily open within 0.01%, with no real differences; the 2 missing days are the first day of data and a day inside the largest data gap. Delay 0 therefore reproduces the backtester's next-day-open entry. Data gaps caused only one skipped trade.

**Trade counts differ slightly from Phase 6.** This study has 36 / 27 development trades (5- / 10-day holds) versus 32 / 24 in notebook 07, because the continuation rule does not use `dist_sma_100`, so trades from October–November 2017 (before that feature's warm-up ended) are included here. Held-out counts (12 / 11) match Phase 6 exactly.

**Waiting did not improve entries.** No delay's entry improvement differed from either base rate by 2 standard errors, in either period or at either holding period. A 1-hour delay was indistinguishable from immediate entry; a 12-hour delay mostly added noise (standard errors 4–5 times larger). A 4-hour delay was worse than the up-trend base rate in all four comparisons (−0.05 to −1.23 standard errors), consistent with prices continuing to rise after a continuation signal, but no single comparison is significant. Base rates were all close to zero, so there is no general intraday drift after 00:00 UTC to exploit.

**Conclusion:** buying at the next day's open, as the backtester already assumes, is as good as any of the pre-registered alternatives. The backtester is not changed.

**Limitations:** BTC only; 12 / 11 held-out trades; only four delays were tested, by design, so the result does not rule out other execution methods (for example limit orders).